# ERGT 01 - Attention Evidence Ladder

This notebook is a short-budget evidence ladder for GeoAttention v2. It is meant to catch implementation errors early and only move to longer runs after the measurement path is clean.

It does four things:

1. Regenerates strict controls and observer reports.
2. Checks relational memory and causal shortest-path geometry.
3. Runs short GeoAttention v2 training controls.
4. Produces a gate result: pass, needs a longer run, incomplete, or needs investigation.

Scope rule: measure first, then bias attention. This notebook does not enable auxiliary loss, the complete ERGT architecture, reasoning evaluation, or intelligence-space evaluation.


## 0. Run Profile

`smoke` is for fast tensor, mask, NaN/Inf, logging, import, and data-path errors. After `smoke` passes, run `sanity`. If that is clean, run `evidence_2000_guarded` for the guarded 2000-step late-window diagnostic.

Training starts only when prepared data exists for the selected profile context length, for example `data/processed/wikitext2_gpt2_ctx64` for `smoke`. In Colab, missing data is prepared automatically by default.


In [ ]:
from pathlib import Path
import copy
import csv
import json
import math
import os
import queue
import subprocess
import sys
import threading
import time
import zipfile

REPO_URL = "https://github.com/jalaljafari2009/ERGT.git"
REPO_BRANCH = "main"
COLAB_REPO_DIR = Path("/content/ERGT")


def running_in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, COLAB_REPO_DIR]
    for candidate in candidates:
        if (candidate / "tests/test_geo_attention.py").exists():
            return candidate.resolve()
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    if running_in_colab():
        if COLAB_REPO_DIR.exists() and not (COLAB_REPO_DIR / ".git").exists():
            raise RuntimeError(
                f"{COLAB_REPO_DIR} exists but is not an ERGT git checkout. "
                "Delete that folder in Colab or restart the runtime."
            )
        if not (COLAB_REPO_DIR / ".git").exists():
            subprocess.run(
                [
                    "git",
                    "clone",
                    "--depth",
                    "1",
                    "--branch",
                    REPO_BRANCH,
                    REPO_URL,
                    str(COLAB_REPO_DIR),
                ],
                check=True,
            )
        PROJECT_ROOT = COLAB_REPO_DIR.resolve()
    else:
        raise RuntimeError(
            "ERGT project root was not found. Run this notebook from the repo root, "
            "or open the Colab GitHub link so the setup cell can clone the repo."
        )

if running_in_colab() and (PROJECT_ROOT / ".git").exists():
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
    commit_result = subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"],
        check=True,
        text=True,
        stdout=subprocess.PIPE,
    )
    GIT_COMMIT = commit_result.stdout.strip()
else:
    GIT_COMMIT = "local"

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import torch
    DEFAULT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEFAULT_DEVICE = "cpu"

RUN_PROFILE = "smoke"  # "smoke", "sanity", "evidence", or "evidence_2000_guarded"
DEVICE = DEFAULT_DEVICE
SEED = 2027
DEFAULT_ALPHA = 0.1

RUN_UNIT_TESTS = True
REGENERATE_OBSERVER_REPORTS = True
RUN_TRAINING = True
PREPARE_DATA_IF_MISSING = running_in_colab()
INSTALL_DATA_DEPS_IF_MISSING = running_in_colab()
STOP_ON_FIRST_FAILURE = True
ABORT_ON_FAILED_OBSERVER_GATE = True
ABORT_ON_FAILED_ATTENTION_GATE = True
EXPORT_REPORT_BUNDLE = True
DOWNLOAD_BUNDLE_IN_COLAB = True
AUTO_SHUTDOWN_COLAB_RUNTIME = True
AUTO_SHUTDOWN_DELAY_SECONDS = 30
INCLUDE_INSTANTANEOUS_CONTROL = True
REPORT_BUNDLE_NAME = "ergt_01_attention_evidence_report_bundle.zip"
TRAJECTORY_TABLE_EVERY_STEPS = 100

PROFILE_SETTINGS = {
    "smoke": {
        "context_length": 64,
        "max_steps": 20,
        "eval_interval": 20,
        "max_eval_batches": 4,
        "batch_size": 4,
        "n_layers": 2,
        "n_heads": 4,
        "hidden_dim": 128,
        "ffn_dim": 512,
        "warmup_steps": 4,
        "alpha": 0.1,
        "alpha_warmup_steps": 4,
        "per_run_timeout_minutes": 6,
    },
    "sanity": {
        "context_length": 128,
        "max_steps": 200,
        "eval_interval": 50,
        "max_eval_batches": 8,
        "batch_size": 8,
        "n_layers": 4,
        "n_heads": 4,
        "hidden_dim": 256,
        "ffn_dim": 1024,
        "warmup_steps": 40,
        "alpha": 0.1,
        "alpha_warmup_steps": 40,
        "per_run_timeout_minutes": 30,
    },
    "evidence": {
        "context_length": 256,
        "max_steps": 1200,
        "eval_interval": 100,
        "max_eval_batches": None,
        "batch_size": 16,
        "n_layers": 6,
        "n_heads": 8,
        "hidden_dim": 512,
        "ffn_dim": 2048,
        "warmup_steps": 200,
        "alpha": 0.1,
        "alpha_warmup_steps": 200,
        "per_run_timeout_minutes": 35,
    },
    "evidence_2000_guarded": {
        "context_length": 256,
        "max_steps": 2000,
        "eval_interval": 100,
        "max_eval_batches": None,
        "batch_size": 16,
        "n_layers": 6,
        "n_heads": 8,
        "hidden_dim": 512,
        "ffn_dim": 2048,
        "warmup_steps": 200,
        "alpha": 0.025,
        "alpha_warmup_steps": 1000,
        "trajectory_windows": [[0, 500], [500, 1000], [1000, 1500], [1500, 2000]],
        "late_window": [1000, 2000],
        "geo_ratio_guard": {
            "early_window": [0, 500],
            "early_max": 0.10,
            "late_window": [1000, 2000],
            "late_max": 0.15,
        },
        "strict_late_window_gate": True,
        "per_run_timeout_minutes": 140,
    },
}

if RUN_PROFILE not in PROFILE_SETTINGS:
    raise ValueError(f"Unknown RUN_PROFILE: {RUN_PROFILE}")

BUDGET = PROFILE_SETTINGS[RUN_PROFILE]
ALPHA = float(BUDGET.get("alpha", DEFAULT_ALPHA))
ALPHA_WARMUP_STEPS = int(BUDGET.get("alpha_warmup_steps", min(BUDGET["warmup_steps"], BUDGET["max_steps"])))
TRAJECTORY_WINDOWS = BUDGET.get("trajectory_windows", [[0, BUDGET["max_steps"]]])
LATE_WINDOW = BUDGET.get("late_window", [max(0, BUDGET["max_steps"] // 2), BUDGET["max_steps"]])
GEO_RATIO_GUARD = BUDGET.get("geo_ratio_guard", {})
STRICT_LATE_WINDOW_GATE = bool(BUDGET.get("strict_late_window_gate", False))
RUN_ID = f"{time.strftime('%Y%m%d_%H%M%S')}_{RUN_PROFILE}"
RUN_ROOT = Path("runs/notebook_01_attention_evidence_ladder") / RUN_ID
CONFIG_DIR = RUN_ROOT / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"project_root={PROJECT_ROOT}")
print(f"git_commit={GIT_COMMIT}")
print(f"device={DEVICE}")
print(f"run_profile={RUN_PROFILE}")
print(f"alpha={ALPHA}")
print(f"alpha_warmup_steps={ALPHA_WARMUP_STEPS}")
print(f"late_window={LATE_WINDOW}")
print(f"run_root={RUN_ROOT}")
print(json.dumps(BUDGET, indent=2, sort_keys=True))

## 1. Utilities


In [ ]:
def read_json(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, sort_keys=True)
        f.write("\n")


def run_cmd(cmd, *, timeout_sec=None, check=True):
    start = time.perf_counter()
    print("\n$ " + " ".join(str(part) for part in cmd))
    env = os.environ.copy()
    env.setdefault("PYTHONUNBUFFERED", "1")
    process = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=PROJECT_ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
        env=env,
    )
    output_lines = []
    line_queue = queue.Queue()

    def _reader():
        try:
            for line in process.stdout or []:
                line_queue.put(line)
        finally:
            line_queue.put(None)

    reader = threading.Thread(target=_reader, daemon=True)
    reader.start()
    reader_done = False
    timed_out = False
    while True:
        try:
            line = line_queue.get(timeout=0.2)
        except queue.Empty:
            line = "__NO_LINE__"
        if line is None:
            reader_done = True
        elif line != "__NO_LINE__":
            output_lines.append(line)
            print(line, end="", flush=True)

        elapsed = time.perf_counter() - start
        if timeout_sec is not None and elapsed > timeout_sec and process.poll() is None:
            timed_out = True
            process.kill()
            print(f"\nTIMEOUT after {elapsed / 60:.1f} min", flush=True)
        if reader_done and process.poll() is not None:
            break

    reader.join(timeout=1.0)
    completed_returncode = "timeout" if timed_out else process.returncode
    elapsed = time.perf_counter() - start
    output = "".join(output_lines)
    print(f"\nreturncode={completed_returncode}, elapsed={elapsed / 60:.2f} min")
    if check and completed_returncode != 0:
        raise RuntimeError(f"Command failed with code {completed_returncode}: {cmd}")
    return {
        "returncode": completed_returncode,
        "elapsed_seconds": elapsed,
        "stdout": output,
    }


def ensure_data_dependencies():
    required = {
        "datasets": "datasets>=2.18",
        "transformers": "transformers>=4.40",
        "tokenizers": "tokenizers>=0.15",
    }
    missing = []
    for module_name, package_spec in required.items():
        try:
            __import__(module_name)
        except Exception:
            missing.append(package_spec)
    if missing:
        run_cmd([sys.executable, "-m", "pip", "install", *missing], timeout_sec=20 * 60)
    else:
        print("Data dependencies already available.")


def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    records = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            records.append(json.loads(line))
    return records


def finite_or_none(value):
    if value is None:
        return None
    try:
        value = float(value)
    except Exception:
        return None
    return value if math.isfinite(value) else None


def export_report_bundle(reason="manual"):
    bundle_path = RUN_ROOT / REPORT_BUNDLE_NAME
    include_paths = [
        Path("notebooks/ERGT_01_Attention_Evidence_Ladder.ipynb"),
        RUN_ROOT / "pretraining_gate_summary.json",
        RUN_ROOT / "training_control_contract.json",
        RUN_ROOT / "training_trajectory_summary.json",
        RUN_ROOT / "training_trajectory_baseline_centered.csv",
        RUN_ROOT / "training_stepwise_loss_geo_ratio_long.csv",
        RUN_ROOT / "training_stepwise_loss_geo_ratio_wide.csv",
        RUN_ROOT / "training_trajectory_baseline_centered.png",
        RUN_ROOT / "training_stepwise_loss_geo_ratio.png",
        RUN_ROOT / "attention_evidence_summary.json",
    ]

    if "CONFIG_DIR" in globals() and CONFIG_DIR.exists():
        include_paths.extend(sorted(CONFIG_DIR.glob("*.json")))

    if "report_paths" in globals():
        include_paths.extend(Path(path) for path in report_paths.values())

    if "run_plan" in globals():
        for item in run_plan:
            output_dir = Path(item["output_dir"])
            include_paths.extend(
                output_dir / name
                for name in [
                    "config.json",
                    "metrics.json",
                    "baseline_results.json",
                    "model_summary.json",
                    "train_log.jsonl",
                    "progress_log.jsonl",
                ]
            )

    filtered_paths = []
    seen = set()
    for path in include_paths:
        path = Path(path)
        if not path.exists() or not path.is_file():
            continue
        if "checkpoints" in path.parts or path.suffix.lower() in {".pt", ".pth", ".ckpt"}:
            continue
        resolved = path.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        filtered_paths.append(path)

    manifest_path = RUN_ROOT / "report_bundle_manifest.json"
    manifest = {
        "reason": reason,
        "bundle_name": REPORT_BUNDLE_NAME,
        "run_profile": RUN_PROFILE,
        "run_root": RUN_ROOT.as_posix(),
        "excluded": ["checkpoints/", "*.pt", "*.pth", "*.ckpt"],
        "files": [path.as_posix() for path in filtered_paths],
    }
    write_json(manifest_path, manifest)
    filtered_paths.append(manifest_path)

    bundle_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in filtered_paths:
            try:
                arcname = path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
            except ValueError:
                arcname = path.name
            archive.write(path, arcname)

    print(f"report_bundle={bundle_path}")
    print(f"files_in_bundle={len(filtered_paths)}")

    if DOWNLOAD_BUNDLE_IN_COLAB:
        try:
            from google.colab import files
            files.download(str(bundle_path))
        except Exception as exc:
            print(f"Colab auto-download skipped: {exc}")
    return bundle_path


def shutdown_colab_runtime_if_requested(reason="completed"):
    if not AUTO_SHUTDOWN_COLAB_RUNTIME:
        print("Auto-shutdown disabled.")
        return
    if not running_in_colab():
        print("Auto-shutdown skipped: not running in Colab.")
        return
    print(
        f"Auto-shutdown enabled: releasing Colab runtime in {AUTO_SHUTDOWN_DELAY_SECONDS}s. "
        f"reason={reason}",
        flush=True,
    )
    time.sleep(max(int(AUTO_SHUTDOWN_DELAY_SECONDS), 0))
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as exc:
        print(f"runtime.unassign failed, killing kernel instead: {exc}", flush=True)
        os.kill(os.getpid(), 9)


def print_table(rows, columns):
    if not rows:
        print("<no rows>")
        return
    widths = []
    for col in columns:
        width = len(col)
        for row in rows:
            width = max(width, len(str(row.get(col, ""))))
        widths.append(width)
    header = " | ".join(col.ljust(width) for col, width in zip(columns, widths))
    print(header)
    print("-+-".join("-" * width for width in widths))
    for row in rows:
        print(" | ".join(str(row.get(col, "")).ljust(width) for col, width in zip(columns, widths)))

## 2. Dataset Gate

This notebook trains on prepared WikiText-2 token blocks. If data is missing in Colab, the notebook installs the data dependencies and prepares the dataset.


In [ ]:
CONTEXT_LENGTH = int(BUDGET["context_length"])
DATA_DIR = Path("data/processed") / f"wikitext2_gpt2_ctx{CONTEXT_LENGTH}"
DATA_READY = (DATA_DIR / "metadata.json").exists()

if not DATA_READY and PREPARE_DATA_IF_MISSING:
    if INSTALL_DATA_DEPS_IF_MISSING:
        ensure_data_dependencies()
    run_cmd([
        sys.executable,
        "scripts/prepare_wikitext2.py",
        "--output-dir",
        DATA_DIR,
        "--tokenizer",
        "gpt2",
        "--context-length",
        str(CONTEXT_LENGTH),
    ], timeout_sec=60 * 60)
    DATA_READY = (DATA_DIR / "metadata.json").exists()

print(f"data_dir={DATA_DIR}")
print(f"data_ready={DATA_READY}")
if DATA_READY:
    print(json.dumps(read_json(DATA_DIR / "metadata.json"), indent=2, sort_keys=True)[:2000])
else:
    print("Training controls will be skipped. Set PREPARE_DATA_IF_MISSING=True or prepare data manually.")

## 3. Fast Unit Smoke


In [ ]:
if RUN_UNIT_TESTS:
    test_targets = [
        "tests/test_geo_attention.py",
        "tests/test_geoattention_v2_report.py",
        "tests/test_control_isolation_audit.py",
        "tests/test_causal_shortest_path_geometry.py",
        "tests/test_relational_memory_observer.py",
        "tests/test_strict_w_controls.py",
    ]
    run_cmd([sys.executable, "-m", "pytest", *test_targets], timeout_sec=10 * 60)
else:
    print("Unit smoke disabled.")

## 4. Regenerate Measurement and Observer Reports

This stage does not train the model. It rebuilds the evidence base: fair controls, relational field observer, resonant response, Phi, reconstruction, memory, causal geometry, and GeoAttention v2 mechanics.


In [ ]:
observer_scripts = [
    "experiments/create_measurement_contract_report.py",
    "experiments/create_strict_w_controls_report.py",
    "experiments/create_control_isolation_audit_report.py",
    "experiments/create_relational_field_observer_report.py",
    "experiments/create_resonant_response_observer_report.py",
    "experiments/create_information_potential_report.py",
    "experiments/create_reconstruction_gate_report.py",
    "experiments/create_relational_memory_observer_report.py",
    "experiments/create_causal_shortest_path_geometry_report.py",
    "experiments/create_geoattention_v2_report.py",
]

if REGENERATE_OBSERVER_REPORTS:
    for script in observer_scripts:
        run_cmd([sys.executable, script], timeout_sec=10 * 60)
else:
    print("Observer regeneration disabled; existing reports will be read.")

In [ ]:
report_paths = {
    "measurement_contracts": "runs/contracts/measurement_contract_report.json",
    "strict_w_controls": "runs/contracts/strict_w_controls_report.json",
    "relational_field": "runs/observers/relational_field_observer_report.json",
    "resonant_response": "runs/observers/resonant_response_observer_report.json",
    "information_potential_phi": "runs/observers/information_potential_report.json",
    "reconstruction_gate": "runs/observers/reconstruction_gate_report.json",
    "relational_memory": "runs/observers/relational_memory_observer_report.json",
    "causal_shortest_path": "runs/observers/causal_shortest_path_geometry_report.json",
    "control_isolation_audit": "runs/geoattention_v2/control_isolation_audit_report.json",
    "geoattention_v2_mechanics": "runs/geoattention_v2/geoattention_v2_report.json",
}

report_rows = []
for name, path in report_paths.items():
    path = Path(path)
    if not path.exists():
        report_rows.append({"report": name, "status": "missing", "failed_checks": "missing", "next": ""})
        continue
    report = read_json(path)
    checks = report.get("checks", {})
    failed = [key for key, value in checks.items() if value is False]
    report_rows.append({
        "report": name,
        "status": report.get("status", ""),
        "failed_checks": ",".join(failed) if failed else "-",
        "next": report.get("next_required_step", ""),
    })

print_table(report_rows, ["report", "status", "failed_checks", "next"])

causal_report = read_json(report_paths["causal_shortest_path"])
isolation_report = read_json(report_paths["control_isolation_audit"])
geo_report = read_json(report_paths["geoattention_v2_mechanics"])
print("\ncausal_future_edges_forbidden=", causal_report.get("checks", {}).get("future_edges_forbidden"))
print("control_isolation_status=", isolation_report.get("status"))
print("geoattention_future_attention_forbidden=", geo_report.get("checks", {}).get("future_attention_forbidden"))
print("geoattention_strict_gate_status=", geo_report.get("strict_gate_status"))

pretraining_required_checks = {
    "all_reports_present": all(Path(path).exists() for path in report_paths.values()),
    "strict_controls_status_pass": read_json(report_paths["strict_w_controls"]).get("status") == "pass",
    "memory_observer_status_pass": read_json(report_paths["relational_memory"]).get("status") == "pass",
    "causal_geometry_status_pass": causal_report.get("status") == "pass",
    "causal_future_edges_forbidden": causal_report.get("checks", {}).get("future_edges_forbidden") is True,
    "causal_future_inputs_forbidden": causal_report.get("checks", {}).get("future_inputs_forbidden") is True,
    "control_isolation_status_pass": isolation_report.get("status") == "pass",
    "control_isolation_same_hidden_inputs": isolation_report.get("checks", {}).get("same_hidden_inputs_for_all_modes") is True,
    "control_isolation_random_no_real_distance_reuse": isolation_report.get("checks", {}).get("no_real_distance_reuse_random") is True,
    "control_isolation_shuffled_no_real_distance_reuse": isolation_report.get("checks", {}).get("no_real_distance_reuse_shuffled") is True,
    "control_isolation_random_no_real_memory_reuse": isolation_report.get("checks", {}).get("no_real_memory_reuse_random") is True,
    "control_isolation_shuffled_no_real_memory_reuse": isolation_report.get("checks", {}).get("no_real_memory_reuse_shuffled") is True,
    "geoattention_mechanics_status_pass": geo_report.get("status") == "pass",
    "geoattention_required_controls_present": geo_report.get("checks", {}).get("required_controls_present") is True,
    "geoattention_stable_causal_geometry_injected": geo_report.get("checks", {}).get("stable_causal_geometry_injected") is True,
    "geoattention_future_attention_forbidden": geo_report.get("checks", {}).get("future_attention_forbidden") is True,
}

observer_failures = []
for row in report_rows:
    if row["status"] != "pass":
        observer_failures.append(f"{row['report']}: status={row['status']}")
    if row["failed_checks"] not in {"-", "", "missing"}:
        observer_failures.append(f"{row['report']}: failed_checks={row['failed_checks']}")
for check_name, value in pretraining_required_checks.items():
    if value is not True:
        observer_failures.append(f"{check_name}={value}")

pretraining_gate = {
    "status": "pass" if not observer_failures else "failed",
    "checks": pretraining_required_checks,
    "failures": observer_failures,
    "note": "geoattention strict training gate is expected to require the training controls below",
}
write_json(RUN_ROOT / "pretraining_gate_summary.json", pretraining_gate)
print("\npretraining_gate_status=", pretraining_gate["status"])
if observer_failures:
    print("pretraining_failures:")
    for failure in observer_failures:
        print("-", failure)
    if EXPORT_REPORT_BUNDLE:
        export_report_bundle(reason="pretraining_gate_failed")
    if ABORT_ON_FAILED_OBSERVER_GATE:
        shutdown_colab_runtime_if_requested(reason="pretraining_gate_failed")
        raise RuntimeError("Pre-training observer gate failed; training controls were not started.")

## 5. Build Short Training Control Configs

The notebook builds these training controls under one shared run root:

- `baseline`
- `alpha_zero`
- `real_memory_d`
- `random_memory_d`
- `shuffled_memory_d`
- `no_memory_real_d`
- `instantaneous_real_d` when `INCLUDE_INSTANTANEOUS_CONTROL=True`

All configs are written to `RUN_ROOT/configs`, and each training output is saved under `RUN_ROOT/<condition>`.


In [ ]:
BASELINE_TEMPLATE = read_json("configs/baseline/phase3_stable_base_seed2027.json")
ERGT_TEMPLATE = read_json("configs/ergt_v1/phase3_stable_base/real_d_alpha_0_1_warmup_cosine_seed2027.json")


def apply_common_budget(config, condition, output_dir):
    config["run"]["phase"] = "notebook_01_attention_evidence_ladder"
    config["run"]["condition"] = condition
    config["run"]["seed"] = SEED
    config["run"]["output_dir"] = Path(output_dir).as_posix()
    config["dataset"]["context_length"] = int(BUDGET["context_length"])
    config["model"]["n_layers"] = BUDGET["n_layers"]
    config["model"]["n_heads"] = BUDGET["n_heads"]
    config["model"]["hidden_dim"] = BUDGET["hidden_dim"]
    config["model"]["ffn_dim"] = BUDGET["ffn_dim"]
    config["training"]["batch_size"] = BUDGET["batch_size"]
    config["training"]["max_steps"] = BUDGET["max_steps"]
    config["training"]["eval_interval"] = BUDGET["eval_interval"]
    config["training"]["checkpoint_interval"] = BUDGET["max_steps"]
    config["training"]["warmup_steps"] = min(BUDGET["warmup_steps"], BUDGET["max_steps"])
    config["training"]["precision"] = BUDGET.get("precision", "auto")
    config["training"]["allow_tf32"] = True
    config["training"]["float32_matmul_precision"] = "high"
    config["training"]["dataloader_num_workers"] = int(BUDGET.get("dataloader_num_workers", 2))
    config["training"]["pin_memory"] = True
    config["training"]["persistent_workers"] = True
    config["training"]["prefetch_factor"] = 2
    if BUDGET.get("max_eval_batches") is None:
        config["training"].pop("max_eval_batches", None)
    else:
        config["training"]["max_eval_batches"] = int(BUDGET["max_eval_batches"])
    return config


def make_baseline_config(condition):
    output_dir = RUN_ROOT / condition
    config = apply_common_budget(copy.deepcopy(BASELINE_TEMPLATE), condition, output_dir)
    return config


def make_ergt_config(condition, distance_mode, alpha):
    output_dir = RUN_ROOT / condition
    config = apply_common_budget(copy.deepcopy(ERGT_TEMPLATE), condition, output_dir)
    attention = config.setdefault("attention", {})
    attention["distance_mode"] = distance_mode
    attention["causal_runtime_distance"] = True
    attention["max_causal_step"] = 1
    attention["gradient_mode"] = "detached_d"
    attention["memory"] = {
        "decay": 0.7,
        "eta": 0.3,
        "gate_floor": 0.05,
        "min_context_edges": 2,
    }
    attention["alpha"] = {
        "mode": "fixed",
        "initial_value": alpha,
        "non_negative": True,
        "warmup_steps": min(ALPHA_WARMUP_STEPS, BUDGET["max_steps"]),
    }
    logging = config.setdefault("logging", {})
    logging["log_geometry_diagnostics"] = True
    logging["train_geometry_diagnostics_interval"] = BUDGET["eval_interval"]
    return config


control_specs = [
    {"condition": "baseline", "kind": "baseline"},
    {"condition": "alpha_zero", "kind": "ergt", "distance_mode": "zero_d", "alpha": 0.0},
    {"condition": "real_memory_d", "kind": "ergt", "distance_mode": "real_stable_causal_d", "alpha": ALPHA},
    {"condition": "random_memory_d", "kind": "ergt", "distance_mode": "random_stable_causal_d", "alpha": ALPHA},
    {"condition": "shuffled_memory_d", "kind": "ergt", "distance_mode": "shuffled_stable_causal_d", "alpha": ALPHA},
    {"condition": "no_memory_real_d", "kind": "ergt", "distance_mode": "no_memory_real_d", "alpha": ALPHA},
]
if INCLUDE_INSTANTANEOUS_CONTROL:
    control_specs.append({
        "condition": "instantaneous_real_d",
        "kind": "ergt",
        "distance_mode": "instantaneous_real_d",
        "alpha": ALPHA,
    })

run_plan = []
for spec in control_specs:
    condition = spec["condition"]
    if spec["kind"] == "baseline":
        config = make_baseline_config(condition)
        script = "experiments/train_baseline.py"
    else:
        config = make_ergt_config(condition, spec["distance_mode"], spec["alpha"])
        script = "experiments/train_ergt_v1.py"
    config_path = CONFIG_DIR / f"{condition}.json"
    write_json(config_path, config)
    run_plan.append({
        **spec,
        "config_path": config_path,
        "output_dir": Path(config["run"]["output_dir"]),
        "script": script,
    })

print_table([
    {
        "condition": item["condition"],
        "kind": item["kind"],
        "distance_mode": item.get("distance_mode", "-"),
        "alpha": item.get("alpha", "-"),
        "config": item["config_path"].as_posix(),
    }
    for item in run_plan
], ["condition", "kind", "distance_mode", "alpha", "config"])


def training_control_signature(config):
    training = config.get("training", {})
    dataset = config.get("dataset", {})
    return {
        "run_seed": config.get("run", {}).get("seed"),
        "dataset": dataset,
        "resolved_data_dir": DATA_DIR.as_posix(),
        "batch_size": training.get("batch_size"),
        "max_steps": training.get("max_steps"),
        "eval_interval": training.get("eval_interval"),
        "max_eval_batches": training.get("max_eval_batches"),
        "warmup_steps": training.get("warmup_steps"),
        "optimizer": training.get("optimizer"),
        "learning_rate": training.get("learning_rate"),
        "precision": training.get("precision"),
        "allow_tf32": training.get("allow_tf32"),
        "dataloader_num_workers": training.get("dataloader_num_workers"),
        "pin_memory": training.get("pin_memory"),
    }


control_configs = {item["condition"]: read_json(item["config_path"]) for item in run_plan}
signatures = {
    condition: training_control_signature(config)
    for condition, config in control_configs.items()
}
reference_signature = signatures["real_memory_d"]
dataset_contract_text = json.dumps(reference_signature["dataset"], sort_keys=True).lower()
training_control_contract = {
    "status": "pending",
    "reference_condition": "real_memory_d",
    "checks": {
        "all_conditions_share_training_data_signature": all(
            signature == reference_signature for signature in signatures.values()
        ),
        "all_conditions_use_same_seed": len({signature["run_seed"] for signature in signatures.values()}) == 1,
        "all_conditions_use_same_resolved_data_dir": len({signature["resolved_data_dir"] for signature in signatures.values()}) == 1,
        "dataset_config_has_no_geometry_terms": not any(
            term in dataset_contract_text
            for term in ["geometry", "distance", "attention", "real_d", "random", "shuffled"]
        ),
    },
    "signatures": signatures,
}
training_control_contract["status"] = "pass" if all(training_control_contract["checks"].values()) else "failed"
write_json(RUN_ROOT / "training_control_contract.json", training_control_contract)
print("training_control_contract_status=", training_control_contract["status"])
print_table(
    [{"check": key, "value": value} for key, value in training_control_contract["checks"].items()],
    ["check", "value"],
)
if training_control_contract["status"] != "pass":
    if EXPORT_REPORT_BUNDLE:
        export_report_bundle(reason="training_control_contract_failed")
    shutdown_colab_runtime_if_requested(reason="training_control_contract_failed")
    raise RuntimeError("Training control contract failed; training controls were not started.")

## 6. Run Training Controls

This stage may take time. Start with `RUN_PROFILE="smoke"`. If it passes, rerun with `RUN_PROFILE="sanity"`, then use `RUN_PROFILE="evidence_2000_guarded"` for the guarded late-window diagnostic.


In [ ]:
training_records = []
per_run_timeout_sec = int(BUDGET["per_run_timeout_minutes"] * 60)

if RUN_TRAINING and DATA_READY:
    for item in run_plan:
        cmd = [
            sys.executable,
            "-u",
            item["script"],
            "--config",
            item["config_path"],
            "--device",
            DEVICE,
        ]
        result = run_cmd(cmd, timeout_sec=per_run_timeout_sec, check=STOP_ON_FIRST_FAILURE)
        training_records.append({
            "condition": item["condition"],
            "returncode": result["returncode"],
            "elapsed_seconds": result["elapsed_seconds"],
        })
else:
    print("Training skipped.")
    print(f"RUN_TRAINING={RUN_TRAINING}, DATA_READY={DATA_READY}")

print_table(training_records, ["condition", "returncode", "elapsed_seconds"])

## 7. Collect Metrics


In [ ]:
def load_training_metrics(item):
    output_dir = Path(item["output_dir"])
    metrics_path = output_dir / ("baseline_results.json" if item["kind"] == "baseline" else "metrics.json")
    if metrics_path.exists():
        metrics = read_json(metrics_path)
        source = metrics_path.as_posix()
    else:
        progress = read_jsonl(output_dir / "progress_log.jsonl")
        validation_records = [row for row in progress if "validation_loss" in row]
        metrics = validation_records[-1] if validation_records else {}
        source = "progress_log_fallback" if metrics else "missing"

    geometry_summary = metrics.get("geometry", {}).get("summary", {}) if isinstance(metrics.get("geometry"), dict) else {}
    return {
        "condition": item["condition"],
        "kind": item["kind"],
        "distance_mode": item.get("distance_mode", "-"),
        "best_validation_loss": finite_or_none(metrics.get("best_validation_loss", metrics.get("validation_loss"))),
        "final_validation_loss": finite_or_none(metrics.get("final_validation_loss", metrics.get("validation_loss"))),
        "perplexity": finite_or_none(metrics.get("perplexity")),
        "geo_to_qk_ratio": finite_or_none(geometry_summary.get("geo_to_qk_ratio")),
        "attention_entropy": finite_or_none(geometry_summary.get("attention_entropy")),
        "tokens_per_second": finite_or_none(metrics.get("average_tokens_per_second")),
        "source": source,
    }


metric_rows = [load_training_metrics(item) for item in run_plan]
print_table(metric_rows, [
    "condition",
    "distance_mode",
    "best_validation_loss",
    "final_validation_loss",
    "geo_to_qk_ratio",
    "attention_entropy",
    "source",
])


def _round_or_none(value, digits=6):
    value = finite_or_none(value)
    return None if value is None else round(value, digits)


def _mean_or_none(values):
    finite = [float(value) for value in values if value is not None and math.isfinite(float(value))]
    return None if not finite else sum(finite) / len(finite)


def _time_normalized_auc(rows, key):
    rows = sorted(
        [row for row in rows if row.get(key) is not None],
        key=lambda row: row["step"],
    )
    if not rows:
        return None
    if len(rows) == 1:
        return rows[0][key]
    area = 0.0
    for left, right in zip(rows, rows[1:]):
        width = right["step"] - left["step"]
        area += width * (left[key] + right[key]) / 2.0
    span = rows[-1]["step"] - rows[0]["step"]
    return None if span <= 0 else area / span


def _load_progress_trajectory(item):
    output_dir = Path(item["output_dir"])
    rows = []
    for record in read_jsonl(output_dir / "progress_log.jsonl"):
        validation_loss = finite_or_none(record.get("validation_loss"))
        if validation_loss is None:
            continue
        rows.append({
            "step": int(record["step"]),
            "condition": item["condition"],
            "distance_mode": item.get("distance_mode", "-"),
            "validation_loss": validation_loss,
            "best_validation_loss": finite_or_none(record.get("best_validation_loss")),
            "train_loss": finite_or_none(record.get("train_loss")),
            "alpha_effective": finite_or_none(record.get("alpha_effective")),
            "alpha_warmup_factor": finite_or_none(record.get("alpha_warmup_factor")),
            "geo_to_qk_ratio": finite_or_none(record.get("geo_to_qk_ratio")),
            "attention_entropy": finite_or_none(record.get("attention_entropy")),
        })
    return rows


trajectory_by_condition = {
    item["condition"]: _load_progress_trajectory(item)
    for item in run_plan
}
baseline_by_step = {
    row["step"]: row["validation_loss"]
    for row in trajectory_by_condition.get("baseline", [])
}
real_by_step = {
    row["step"]: row["validation_loss"]
    for row in trajectory_by_condition.get("real_memory_d", [])
}

trajectory_rows = []
for condition, rows in trajectory_by_condition.items():
    for row in rows:
        baseline_loss = baseline_by_step.get(row["step"])
        real_loss = real_by_step.get(row["step"])
        validation_loss = row["validation_loss"]
        improvement = None if baseline_loss is None else baseline_loss - validation_loss
        trajectory_rows.append({
            **row,
            "baseline_loss_same_step": baseline_loss,
            "real_loss_same_step": real_loss,
            "baseline_centered_improvement": improvement,
            "delta_loss_vs_baseline": None if baseline_loss is None else validation_loss - baseline_loss,
            "delta_loss_vs_real": None if real_loss is None else validation_loss - real_loss,
        })

trajectory_condition_summary = {}
for condition in trajectory_by_condition:
    rows = [row for row in trajectory_rows if row["condition"] == condition]
    rows = sorted(rows, key=lambda row: row["step"])
    improvements = [row["baseline_centered_improvement"] for row in rows]
    trajectory_condition_summary[condition] = {
        "eval_points": len(rows),
        "final_step": rows[-1]["step"] if rows else None,
        "final_validation_loss": rows[-1]["validation_loss"] if rows else None,
        "final_baseline_centered_improvement": rows[-1]["baseline_centered_improvement"] if rows else None,
        "mean_baseline_centered_improvement": _mean_or_none(improvements),
        "best_baseline_centered_improvement": max([v for v in improvements if v is not None], default=None),
        "time_normalized_auc_baseline_centered_improvement": _time_normalized_auc(
            rows,
            "baseline_centered_improvement",
        ),
    }


def _rows_for_window(rows, window):
    start, end = int(window[0]), int(window[1])
    return [row for row in rows if start <= row["step"] <= end]


def _summarize_window_rows(rows):
    rows = sorted(rows, key=lambda row: row["step"])
    improvements = [row["baseline_centered_improvement"] for row in rows]
    deltas_vs_real = [row["delta_loss_vs_real"] for row in rows]
    geo_ratios = [row["geo_to_qk_ratio"] for row in rows]
    finite_geo_ratios = [value for value in geo_ratios if value is not None and math.isfinite(float(value))]
    return {
        "eval_points": len(rows),
        "first_step": rows[0]["step"] if rows else None,
        "last_step": rows[-1]["step"] if rows else None,
        "mean_validation_loss": _mean_or_none([row["validation_loss"] for row in rows]),
        "mean_baseline_centered_improvement": _mean_or_none(improvements),
        "time_normalized_auc_baseline_centered_improvement": _time_normalized_auc(
            rows,
            "baseline_centered_improvement",
        ),
        "mean_delta_loss_vs_real": _mean_or_none(deltas_vs_real),
        "time_normalized_auc_delta_loss_vs_real": _time_normalized_auc(
            rows,
            "delta_loss_vs_real",
        ),
        "mean_geo_to_qk_ratio": _mean_or_none(geo_ratios),
        "max_geo_to_qk_ratio": max(finite_geo_ratios, default=None),
    }


def _window_condition_summaries(window):
    return {
        condition: _summarize_window_rows(
            _rows_for_window(
                [row for row in trajectory_rows if row["condition"] == condition],
                window,
            )
        )
        for condition in trajectory_by_condition
    }


def _window_name(window):
    return f"{int(window[0])}_{int(window[1])}"


def _metric(summary, condition, key):
    value = summary.get(condition, {}).get(key)
    return value if value is not None and math.isfinite(float(value)) else None


trajectory_window_summary = {
    _window_name(window): _window_condition_summaries(window)
    for window in TRAJECTORY_WINDOWS
}
late_window_summary = _window_condition_summaries(LATE_WINDOW)
late_real_improvement = _metric(
    late_window_summary,
    "real_memory_d",
    "time_normalized_auc_baseline_centered_improvement",
)

trajectory_control_warnings = []
real_auc = trajectory_condition_summary.get("real_memory_d", {}).get(
    "time_normalized_auc_baseline_centered_improvement"
)
for control in ["random_memory_d", "shuffled_memory_d", "no_memory_real_d", "instantaneous_real_d"]:
    control_auc = trajectory_condition_summary.get(control, {}).get(
        "time_normalized_auc_baseline_centered_improvement"
    )
    if real_auc is not None and control_auc is not None and control_auc > real_auc:
        trajectory_control_warnings.append(
            f"{control}_auc_above_real_memory_d: {control_auc:.6f} > {real_auc:.6f}"
        )

trajectory_required_conditions = [
    "baseline",
    "alpha_zero",
    "real_memory_d",
    "random_memory_d",
    "shuffled_memory_d",
    "no_memory_real_d",
]
if INCLUDE_INSTANTANEOUS_CONTROL:
    trajectory_required_conditions.append("instantaneous_real_d")

late_window_checks = {
    "late_window_has_required_conditions": all(
        late_window_summary.get(condition, {}).get("eval_points", 0) > 0
        for condition in trajectory_required_conditions
    ),
    "real_late_beats_baseline": late_real_improvement is not None and late_real_improvement > 0.0,
}
for control in ["random_memory_d", "shuffled_memory_d", "no_memory_real_d", "instantaneous_real_d"]:
    if control not in trajectory_by_condition:
        continue
    control_delta_vs_real = _metric(
        late_window_summary,
        control,
        "time_normalized_auc_delta_loss_vs_real",
    )
    late_window_checks[f"real_late_beats_{control}"] = (
        control_delta_vs_real is not None and control_delta_vs_real > 0.0
    )
for control in ["random_memory_d", "shuffled_memory_d"]:
    control_improvement = _metric(
        late_window_summary,
        control,
        "time_normalized_auc_baseline_centered_improvement",
    )
    late_window_checks[f"{control}_late_not_better_than_baseline"] = (
        control_improvement is not None and control_improvement <= 0.0
    )

geo_ratio_guard_summary = {}
geo_ratio_guard_checks = {}
if GEO_RATIO_GUARD:
    for label, window_key, max_key in [
        ("early", "early_window", "early_max"),
        ("late", "late_window", "late_max"),
    ]:
        window = GEO_RATIO_GUARD.get(window_key)
        allowed = GEO_RATIO_GUARD.get(max_key)
        if window is None or allowed is None:
            continue
        summary = _window_condition_summaries(window)
        observed = []
        per_condition = {}
        for condition, condition_summary in summary.items():
            if condition == "baseline":
                continue
            value = condition_summary.get("max_geo_to_qk_ratio")
            per_condition[condition] = value
            if value is not None:
                observed.append(float(value))
        observed_max = max(observed, default=None)
        geo_ratio_guard_summary[label] = {
            "window": window,
            "max_allowed": allowed,
            "observed_max": observed_max,
            "per_condition_max": per_condition,
        }
        geo_ratio_guard_checks[f"{label}_geo_to_qk_ratio_within_guard"] = (
            observed_max is not None and observed_max <= float(allowed)
        )

for key, value in late_window_checks.items():
    if value is False:
        trajectory_control_warnings.append(f"late_window_check_failed: {key}")
for key, value in geo_ratio_guard_checks.items():
    if value is False:
        trajectory_control_warnings.append(f"geo_ratio_guard_failed: {key}")

trajectory_summary = {
    "status": "pass" if trajectory_rows else "missing",
    "definition": {
        "baseline_centered_improvement": "baseline_validation_loss_same_step - condition_validation_loss",
        "delta_loss_vs_real": "condition_validation_loss - real_memory_d_validation_loss at the same step",
        "geo_to_qk_ratio": "mean absolute geometry bias term divided by mean absolute QK logits; baseline has no geometry term",
        "positive_is_better": True,
        "paired_by_eval_step": True,
        "table_every_steps": TRAJECTORY_TABLE_EVERY_STEPS,
        "window_inclusion": "start <= step <= end",
    },
    "condition_summary": trajectory_condition_summary,
    "window_summary": trajectory_window_summary,
    "late_window": LATE_WINDOW,
    "late_window_summary": late_window_summary,
    "late_window_checks": late_window_checks,
    "geo_ratio_guard": geo_ratio_guard_summary,
    "geo_ratio_guard_checks": geo_ratio_guard_checks,
    "stepwise_outputs": {
        "long_csv": (RUN_ROOT / "training_stepwise_loss_geo_ratio_long.csv").as_posix(),
        "wide_csv": (RUN_ROOT / "training_stepwise_loss_geo_ratio_wide.csv").as_posix(),
        "plot": (RUN_ROOT / "training_stepwise_loss_geo_ratio.png").as_posix(),
    },
    "control_warnings": trajectory_control_warnings,
}
write_json(RUN_ROOT / "training_trajectory_summary.json", trajectory_summary)

trajectory_csv_path = RUN_ROOT / "training_trajectory_baseline_centered.csv"
trajectory_csv_path.parent.mkdir(parents=True, exist_ok=True)
trajectory_fields = [
    "step",
    "condition",
    "distance_mode",
    "validation_loss",
    "baseline_loss_same_step",
    "real_loss_same_step",
    "baseline_centered_improvement",
    "delta_loss_vs_baseline",
    "delta_loss_vs_real",
    "alpha_effective",
    "alpha_warmup_factor",
    "geo_to_qk_ratio",
    "attention_entropy",
]
with trajectory_csv_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=trajectory_fields)
    writer.writeheader()
    for row in sorted(trajectory_rows, key=lambda item: (item["step"], item["condition"])):
        writer.writerow({field: row.get(field) for field in trajectory_fields})

stepwise_long_csv_path = RUN_ROOT / "training_stepwise_loss_geo_ratio_long.csv"
stepwise_wide_csv_path = RUN_ROOT / "training_stepwise_loss_geo_ratio_wide.csv"
stepwise_core_conditions = [
    "baseline",
    "real_memory_d",
    "random_memory_d",
    "shuffled_memory_d",
    "no_memory_real_d",
    "instantaneous_real_d",
    "alpha_zero",
]
stepwise_conditions = [
    item["condition"] for item in run_plan
    if item["condition"] in stepwise_core_conditions
]
stepwise_steps = sorted({row["step"] for row in trajectory_rows})
stepwise_long_fields = [
    "step",
    "condition",
    "validation_loss",
    "baseline_centered_improvement",
    "delta_loss_vs_real",
    "geo_to_qk_ratio",
    "alpha_effective",
    "alpha_warmup_factor",
    "attention_entropy",
]
with stepwise_long_csv_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=stepwise_long_fields)
    writer.writeheader()
    for row in sorted(trajectory_rows, key=lambda item: (item["step"], item["condition"])):
        if row["condition"] not in stepwise_conditions:
            continue
        writer.writerow({field: row.get(field) for field in stepwise_long_fields})

step_condition_lookup = {
    (row["step"], row["condition"]): row
    for row in trajectory_rows
}
stepwise_wide_fields = ["step"]
for condition in stepwise_conditions:
    stepwise_wide_fields.extend([
        f"{condition}__validation_loss",
        f"{condition}__geo_to_qk_ratio",
        f"{condition}__baseline_centered_improvement",
        f"{condition}__delta_loss_vs_real",
        f"{condition}__alpha_effective",
    ])
with stepwise_wide_csv_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=stepwise_wide_fields)
    writer.writeheader()
    for step in stepwise_steps:
        wide_row = {"step": step}
        for condition in stepwise_conditions:
            row = step_condition_lookup.get((step, condition), {})
            wide_row[f"{condition}__validation_loss"] = row.get("validation_loss")
            wide_row[f"{condition}__geo_to_qk_ratio"] = row.get("geo_to_qk_ratio")
            wide_row[f"{condition}__baseline_centered_improvement"] = row.get("baseline_centered_improvement")
            wide_row[f"{condition}__delta_loss_vs_real"] = row.get("delta_loss_vs_real")
            wide_row[f"{condition}__alpha_effective"] = row.get("alpha_effective")
        writer.writerow(wide_row)

trajectory_display_rows = []
for row in sorted(trajectory_rows, key=lambda item: (item["step"], item["condition"])):
    step = row["step"]
    if step != 1 and step != BUDGET["max_steps"] and step % TRAJECTORY_TABLE_EVERY_STEPS != 0:
        continue
    trajectory_display_rows.append({
        "step": step,
        "condition": row["condition"],
        "val_loss": _round_or_none(row["validation_loss"]),
        "base_centered": _round_or_none(row["baseline_centered_improvement"]),
        "delta_vs_real": _round_or_none(row["delta_loss_vs_real"]),
        "geo/qk": _round_or_none(row["geo_to_qk_ratio"], digits=4),
    })

print("\nBaseline-centered trajectory: positive base_centered means better than baseline at the same step.")
print_table(
    trajectory_display_rows,
    ["step", "condition", "val_loss", "base_centered", "delta_vs_real", "geo/qk"],
)
stepwise_display_rows = []
for row in sorted(trajectory_rows, key=lambda item: (item["step"], item["condition"])):
    step = row["step"]
    if row["condition"] not in stepwise_conditions:
        continue
    if step != 1 and step != BUDGET["max_steps"] and step % TRAJECTORY_TABLE_EVERY_STEPS != 0:
        continue
    stepwise_display_rows.append({
        "step": step,
        "condition": row["condition"],
        "val_loss": _round_or_none(row["validation_loss"]),
        "geo/qk": _round_or_none(row["geo_to_qk_ratio"], digits=4),
        "alpha": _round_or_none(row["alpha_effective"], digits=4),
        "alpha_warmup": _round_or_none(row["alpha_warmup_factor"], digits=4),
        "base_centered": _round_or_none(row["baseline_centered_improvement"]),
        "delta_vs_real": _round_or_none(row["delta_loss_vs_real"]),
    })

print("\nStepwise loss and geo/qk table: baseline has no geometry ratio; positive base_centered means better than baseline.")
print_table(
    stepwise_display_rows,
    ["step", "condition", "val_loss", "geo/qk", "alpha", "alpha_warmup", "base_centered", "delta_vs_real"],
)
late_window_display_rows = []
for condition in trajectory_required_conditions:
    condition_summary = late_window_summary.get(condition, {})
    late_window_display_rows.append({
        "condition": condition,
        "eval_points": condition_summary.get("eval_points"),
        "auc_vs_base": _round_or_none(
            condition_summary.get("time_normalized_auc_baseline_centered_improvement")
        ),
        "auc_delta_vs_real": _round_or_none(
            condition_summary.get("time_normalized_auc_delta_loss_vs_real")
        ),
        "mean_val_loss": _round_or_none(condition_summary.get("mean_validation_loss")),
        "max_geo/qk": _round_or_none(condition_summary.get("max_geo_to_qk_ratio"), digits=4),
    })
print(f"\nLate-window summary {LATE_WINDOW}: positive auc_vs_base means better than baseline; positive auc_delta_vs_real means worse than real.")
print_table(
    late_window_display_rows,
    ["condition", "eval_points", "auc_vs_base", "auc_delta_vs_real", "mean_val_loss", "max_geo/qk"],
)
print_table(
    [{"check": key, "value": value} for key, value in late_window_checks.items()],
    ["check", "value"],
)
print(f"trajectory_summary_path={RUN_ROOT / 'training_trajectory_summary.json'}")
print(f"trajectory_csv_path={trajectory_csv_path}")
print(f"stepwise_long_csv_path={stepwise_long_csv_path}")
print(f"stepwise_wide_csv_path={stepwise_wide_csv_path}")
if trajectory_control_warnings:
    print("trajectory_control_warnings:")
    for warning in trajectory_control_warnings:
        print("-", warning)

trajectory_plot_path = RUN_ROOT / "training_trajectory_baseline_centered.png"
if trajectory_rows:
    try:
        import matplotlib.pyplot as plt

        plt.figure(figsize=(10, 5))
        for item in run_plan:
            condition = item["condition"]
            rows = sorted(
                [
                    row for row in trajectory_rows
                    if row["condition"] == condition and row["baseline_centered_improvement"] is not None
                ],
                key=lambda row: row["step"],
            )
            if not rows:
                continue
            linestyle = "--" if condition == "baseline" else "-"
            linewidth = 2.5 if condition == "real_memory_d" else 1.5
            plt.plot(
                [row["step"] for row in rows],
                [row["baseline_centered_improvement"] for row in rows],
                marker="o",
                linewidth=linewidth,
                linestyle=linestyle,
                label=condition,
            )
        plt.axhline(0.0, color="black", linewidth=1.0, alpha=0.6)
        plt.xlabel("training step")
        plt.ylabel("baseline loss - condition loss")
        plt.title("Baseline-centered validation trajectory")
        plt.grid(True, alpha=0.25)
        plt.legend(loc="best", fontsize=8)
        plt.tight_layout()
        plt.savefig(trajectory_plot_path, dpi=160, bbox_inches="tight")
        plt.show()
        print(f"trajectory_plot_path={trajectory_plot_path}")
    except Exception as exc:
        print(f"Trajectory plot skipped: {exc}")

stepwise_plot_path = RUN_ROOT / "training_stepwise_loss_geo_ratio.png"
if trajectory_rows:
    try:
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
        for condition in stepwise_conditions:
            rows = sorted(
                [row for row in trajectory_rows if row["condition"] == condition],
                key=lambda row: row["step"],
            )
            if not rows:
                continue
            linewidth = 2.5 if condition == "real_memory_d" else 1.5
            linestyle = "--" if condition in {"baseline", "alpha_zero"} else "-"
            axes[0].plot(
                [row["step"] for row in rows],
                [row["validation_loss"] for row in rows],
                marker="o",
                linewidth=linewidth,
                linestyle=linestyle,
                label=condition,
            )
            geo_rows = [
                row for row in rows
                if row.get("geo_to_qk_ratio") is not None
            ]
            if geo_rows:
                axes[1].plot(
                    [row["step"] for row in geo_rows],
                    [row["geo_to_qk_ratio"] for row in geo_rows],
                    marker="o",
                    linewidth=linewidth,
                    linestyle=linestyle,
                    label=condition,
                )
        axes[0].set_ylabel("validation loss")
        axes[0].set_title("Validation loss by condition; lower is better")
        axes[0].grid(True, alpha=0.25)
        axes[0].legend(loc="best", fontsize=8)
        axes[1].set_xlabel("training step")
        axes[1].set_ylabel("geo_to_qk_ratio")
        axes[1].set_title("Geometry/QK ratio by condition; baseline has no geometry term")
        axes[1].grid(True, alpha=0.25)
        if GEO_RATIO_GUARD:
            early_max = GEO_RATIO_GUARD.get("early_max")
            late_max = GEO_RATIO_GUARD.get("late_max")
            if early_max is not None:
                axes[1].axhline(float(early_max), color="tab:orange", linestyle=":", linewidth=1.0, label="early guard")
            if late_max is not None and late_max != early_max:
                axes[1].axhline(float(late_max), color="tab:red", linestyle=":", linewidth=1.0, label="late guard")
        axes[1].legend(loc="best", fontsize=8)
        fig.tight_layout()
        fig.savefig(stepwise_plot_path, dpi=160, bbox_inches="tight")
        plt.show()
        print(f"stepwise_plot_path={stepwise_plot_path}")
    except Exception as exc:
        print(f"Stepwise loss/geo plot skipped: {exc}")

## 8. Attention Evidence Gate

For short runs, this gate is not a final scientific claim. A clean `smoke` or `sanity` result means the mechanics are healthy and a guarded evidence run is justified. With `RUN_PROFILE="evidence_2000_guarded"`, the decision is based on the late window, not on whole-run best validation loss.


In [ ]:
scores = {
    row["condition"]: row["best_validation_loss"]
    for row in metric_rows
    if row["best_validation_loss"] is not None
}

required_conditions = [
    "baseline",
    "alpha_zero",
    "real_memory_d",
    "random_memory_d",
    "shuffled_memory_d",
    "no_memory_real_d",
]
if INCLUDE_INSTANTANEOUS_CONTROL:
    required_conditions.append("instantaneous_real_d")


def lower_is_better(left, right):
    if left not in scores or right not in scores:
        return None
    return scores[left] < scores[right]


def losses_close(left, right, tolerance=1e-8):
    if left not in scores or right not in scores:
        return None
    return abs(scores[left] - scores[right]) <= tolerance


mechanics_checks = {
    "observer_reports_present": all(Path(path).exists() for path in report_paths.values()),
    "strict_controls_pass": read_json(report_paths["strict_w_controls"]).get("status") == "pass",
    "control_isolation_pass": read_json(report_paths["control_isolation_audit"]).get("status") == "pass",
    "training_control_contract_pass": read_json(RUN_ROOT / "training_control_contract.json").get("status") == "pass",
    "causal_future_edges_forbidden": read_json(report_paths["causal_shortest_path"]).get("checks", {}).get("future_edges_forbidden") is True,
    "attention_future_forbidden": read_json(report_paths["geoattention_v2_mechanics"]).get("checks", {}).get("future_attention_forbidden") is True,
    "required_training_metrics_present": all(condition in scores for condition in required_conditions),
    "training_losses_finite": all(
        row["best_validation_loss"] is not None and row["final_validation_loss"] is not None
        for row in metric_rows
    ),
    "alpha_zero_matches_baseline": losses_close("alpha_zero", "baseline"),
}

comparison_checks = {
    "real_beats_baseline": lower_is_better("real_memory_d", "baseline"),
    "real_beats_alpha_zero": lower_is_better("real_memory_d", "alpha_zero"),
    "real_beats_random": lower_is_better("real_memory_d", "random_memory_d"),
    "real_beats_shuffled": lower_is_better("real_memory_d", "shuffled_memory_d"),
    "real_beats_no_memory": lower_is_better("real_memory_d", "no_memory_real_d"),
}
if INCLUDE_INSTANTANEOUS_CONTROL:
    comparison_checks["real_beats_instantaneous"] = lower_is_better("real_memory_d", "instantaneous_real_d")

late_window_gate_checks = {}
if STRICT_LATE_WINDOW_GATE:
    for key, value in trajectory_summary.get("late_window_checks", {}).items():
        late_window_gate_checks[f"late_window_{key}"] = value is True
    for key, value in trajectory_summary.get("geo_ratio_guard_checks", {}).items():
        late_window_gate_checks[f"geo_ratio_{key}"] = value is True

gate_checks = {**mechanics_checks, **comparison_checks, **late_window_gate_checks}
mechanics_failures = [
    f"{key}={value}" for key, value in mechanics_checks.items() if value is not True
]
comparison_warnings = [
    f"{key}={value}" for key, value in comparison_checks.items() if value is not True
]
late_window_failures = [
    f"{key}={value}" for key, value in late_window_gate_checks.items() if value is not True
]

if not RUN_TRAINING or not DATA_READY:
    gate_status = "not_run_training_controls"
elif any(value is None for value in mechanics_checks.values()):
    gate_status = "incomplete"
elif mechanics_failures:
    gate_status = "needs_investigation"
elif RUN_PROFILE in {"smoke", "sanity"}:
    gate_status = "mechanics_pass_needs_evidence_run"
elif STRICT_LATE_WINDOW_GATE and late_window_failures:
    gate_status = "needs_investigation"
elif not STRICT_LATE_WINDOW_GATE and any(value is None for value in comparison_checks.values()):
    gate_status = "incomplete"
elif not STRICT_LATE_WINDOW_GATE and comparison_warnings:
    gate_status = "needs_investigation"
else:
    gate_status = "attention_evidence_pass"

gate_rows = [{"check": key, "value": value} for key, value in gate_checks.items()]
print_table(gate_rows, ["check", "value"])
print("\n" + f"gate_status={gate_status}")

attention_gate_failures = list(mechanics_failures)
if STRICT_LATE_WINDOW_GATE:
    attention_gate_failures.extend(late_window_failures)
elif RUN_PROFILE == "evidence":
    attention_gate_failures.extend(comparison_warnings)

if comparison_warnings:
    print("comparison_warnings:")
    for warning in comparison_warnings:
        print("-", warning)
if late_window_failures:
    print("late_window_failures:")
    for failure in late_window_failures:
        print("-", failure)

summary = {
    "run_profile": RUN_PROFILE,
    "run_root": RUN_ROOT.as_posix(),
    "budget": BUDGET,
    "device": DEVICE,
    "git_commit": GIT_COMMIT,
    "alpha": ALPHA,
    "gate_status": gate_status,
    "mechanics_checks": mechanics_checks,
    "comparison_checks": comparison_checks,
    "late_window_gate_checks": late_window_gate_checks,
    "gate_checks": gate_checks,
    "gate_failures": attention_gate_failures,
    "comparison_warnings": comparison_warnings,
    "late_window_failures": late_window_failures,
    "metrics": metric_rows,
    "trajectory_summary": trajectory_summary if "trajectory_summary" in globals() else {},
}
summary_path = RUN_ROOT / "attention_evidence_summary.json"
write_json(summary_path, summary)
print(f"summary_path={summary_path}")

if EXPORT_REPORT_BUNDLE:
    export_report_bundle(reason=gate_status)

if gate_status in {"needs_investigation", "incomplete", "not_run_training_controls"}:
    if attention_gate_failures:
        print("attention_gate_failures:")
        for failure in attention_gate_failures:
            print("-", failure)
    if ABORT_ON_FAILED_ATTENTION_GATE:
        shutdown_colab_runtime_if_requested(reason=gate_status)
        raise RuntimeError(f"Attention evidence gate failed: {gate_status}")

shutdown_colab_runtime_if_requested(reason=gate_status)


## 9. Decision Rule

- `mechanics_pass_needs_evidence_run`: smoke/sanity mechanics are clean. Comparison warnings are informational at this profile; rerun with `RUN_PROFILE="evidence_2000_guarded"` when ready.
- `attention_evidence_pass`: the guarded late-window checks passed; run multi-seed confirmation before considering the auxiliary physics-inspired loss gate.
- `needs_investigation`: inspect alpha, geo/qk ratio, normalization, memory decay/eta, control fairness, or training stability before longer runs.
- `not_run_training_controls`: prepare data or enable training controls, then rerun from the smoke profile.

The notebook exports only lightweight review files: configs, metrics, progress logs, train logs, observer reports, and summaries. Checkpoints are excluded.

Fixed output bundle name:

`ergt_01_attention_evidence_report_bundle.zip`

Default local review path after Colab download:

`C:\Users\Administrator\Downloads\ergt_01_attention_evidence_report_bundle.zip`
